In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.feature import hog

from sklearn.base import clone
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, classification_report,
)

%pip install "opencv-python<5"
from src.preprocess import preprocess_kdef_image

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2.1 Indexing KDEF images

In [2]:
EMOTIONS = {
    "AF": "afraid",
    "AN": "angry",
    "DI": "disgust",
    "HA": "happy",
    "NE": "neutral",
    "SA": "sad",
    "SU": "surprise",
}

POSES = {
    "FL": "full_left",
    "HL": "half_left",
    "S":  "straight",
    "HR": "half_right",
    "FR": "full_right",
}

# KDEF emotion label to FER2013 label map
KDEF_TO_FER = {
    "angry": 0, "disgust": 1, "afraid": 2, "happy": 3,
    "sad": 4, "surprise": 5, "neutral": 6,
}

emotionDict = {
    0: 'Angry', 1: 'Disgust', 2: 'Fear', 3: 'Happy',
    4: 'Sad', 5: 'Surprise', 6: 'Neutral',
}
class_names = [emotionDict[i] for i in range(7)]

ROOT = Path("data/KDEF_and_AKDEF/KDEF/")

records = []
for img_path in ROOT.rglob("*.JPG"):
    stem = img_path.stem
    
    for emotion_code in EMOTIONS:
        
        if stem[4:].startswith(emotion_code):
            pose_code = stem[4 + len(emotion_code):]
            
            if pose_code in POSES:
                records.append({
                    "path": str(img_path),
                    "acquisition": stem[0],   # session A or B
                    "subject": stem[1:4],  
                    "emotion": EMOTIONS[emotion_code],
                    "pose": POSES[pose_code],
                })
                
            break

df = pd.DataFrame(records)
print("All KDEF images:", len(df))

All KDEF images: 4898


## 2.2 80/20 Split frontal images
Also is subject independent! 

In [ ]:
with open("auxFiles/kdef_subject_split.pkl", "rb") as f:
    split_data = pickle.load(f)

train_subjects = set(split_data["train_subjects"])
test_subjects = set(split_data["test_subjects"])

# Selection filter for frontal images
frontal = df[df["pose"] == "straight"]

train_df = frontal[frontal["subject"].isin(train_subjects)].reset_index(drop=True)
test_df = frontal[frontal["subject"].isin(test_subjects)].reset_index(drop=True)

# Subject independence check: no identity may be in both sets
assert train_subjects.isdisjoint(test_subjects)
assert set(train_df["subject"]).isdisjoint(set(test_df["subject"]))

n_subj = len(train_subjects) + len(test_subjects)
print(f"Train subjects: {len(train_subjects)} ({len(train_subjects) / n_subj:.0%})")
print(f"Test subjects : {len(test_subjects)} ({len(test_subjects) / n_subj:.0%})")
print("Train images  :", len(train_df))
print("Test images   :", len(test_df))
print()
print(pd.crosstab(
    frontal["emotion"],
    frontal["subject"].isin(train_subjects).map({True: "train", False: "test"})))

Train subjects: 56 (80%)
Test subjects : 14 (20%)
Train images  : 784
Test images   : 196

subject   test  train
emotion              
afraid      28    112
angry       28    112
disgust     28    112
happy       28    112
neutral     28    112
sad         28    112
surprise    28    112
